# M7 · Data-quality validation

**Goal:** run the 8 checks in `sql/99_validation_suite.sql` via `accent_fleet.monitoring.run_validation_suite()` and confirm they all pass.

**Exit criterion:** `report.all_passed == True`.


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs

In [ ]:
from accent_fleet.db.sql_loader import load_sql
print(load_sql('99_validation_suite.sql')[:800], '...')


## 3. Execute

In [ ]:
from accent_fleet.monitoring import run_validation_suite
report = run_validation_suite()
print(f'all_passed = {report.all_passed}')
for c in report.checks:
    print(c)


## 4. Inspect

In [ ]:
# Freshness sanity
from accent_fleet.monitoring.quality import check_freshness
f = check_freshness()
print(f)
